In [20]:
import os
import glob
import cv2
import numpy as np

# 1. Initialize OpenCV YuNet Detector and SFace Recognizer
#detector = cv2.FaceDetectorYN.create("Datasets/face_detection_yunet_2023mar.onnx", "", (0, 0))
detector = cv2.FaceDetectorYN.create(
    model="Datasets/face_detection_yunet_2023mar.onnx",
    config="",
    input_size=(0, 0),
    score_threshold=0.45,   # <-- Drop from 0.9 to 0.45 to catch blurry faces
    nms_threshold=0.3
)

recognizer = cv2.FaceRecognizerSF.create("Datasets/face_recognition_sface_2021dec.onnx", "")

def extract_embedding(img_path):
    """Loads an image (handles high-res downscaling) and extracts SFace embedding."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    h_orig, w_orig, _ = img.shape
    target_width = 1000
    
    # Safely downscale high-res images (e.g., 4032x3024) for YuNet detection
    if w_orig > target_width:
        scale_factor = target_width / float(w_orig)
        img_small = cv2.resize(img, (target_width, int(h_orig * scale_factor)))
        detector.setInputSize((img_small.shape[1], img_small.shape[0]))
        _, faces = detector.detect(img_small)
        if faces is not None:
            faces[:, :14] = faces[:, :14] / scale_factor
    else:
        detector.setInputSize((w_orig, h_orig))
        _, faces = detector.detect(img)

    if faces is None:
        return None

    face_aligned = recognizer.alignCrop(img, faces)
    return recognizer.feature(face_aligned)

# 2. Load Folder Structure into Memory
def load_face_templates(dataset_dir):
    """
    Scans folders, averages embeddings for people with multiple photos,
    and returns a dictionary: { "Person_Name": embedding_array }
    """
    templates = {}
    valid_extensions = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.JPG', '*.JPEG', '*.PNG')
    
    if not os.path.exists(dataset_dir):
        print(f"Error: Directory '{dataset_dir}' not found.")
        return templates

    print(f"Indexing dataset folder: {dataset_dir}")
    
    for person_name in os.listdir(dataset_dir):
        person_folder = os.path.join(dataset_dir, person_name)
        if not os.path.isdir(person_folder):
            continue
            
        image_paths = []
        for ext in valid_extensions:
            image_paths.extend(glob.glob(os.path.join(person_folder, ext)))
            
        if not image_paths:
            continue
            
        person_embeddings = []
        for img_path in image_paths:
            embedding = extract_embedding(img_path)
            if embedding is not None:
                person_embeddings.append(embedding)
                
        if person_embeddings:
            # Average vectors for robust multi-photo templates
            master_embedding = np.mean(person_embeddings, axis=0, dtype=np.float32)
            cv2.normalize(master_embedding, master_embedding)
            templates[person_name] = master_embedding
            print(f" -> Indexed: {person_name} ({len(person_embeddings)} photos)")
            
    print("--- Indexing Complete ---\n")
    return templates

# 3. Match a Query Image Against Loaded Templates
def identify_face_from_memory(query_img_path, templates):
    """Compares a new face image against the dictionary of loaded templates."""
    query_feat = extract_embedding(query_img_path)
    if query_feat is None:
        print("Could not detect a face in the query image.")
        return "Unknown"

    best_match_name = "Unknown"
    highest_score = -1.0
    SFACE_THRESHOLD = 0.363 # Official OpenCV Cosine Threshold

    # Loop through our dictionary keys and vector arrays
    for person_name, db_feat in templates.items():
        score = recognizer.match(query_feat, db_feat, cv2.FaceRecognizerSF_FR_COSINE)
        
        if score > highest_score:
            highest_score = score
            if score >= SFACE_THRESHOLD:
                best_match_name = person_name

    print(f"Result: Match found -> {best_match_name} (Score: {highest_score:.4f})")
    return best_match_name


In [21]:
# 1. Load the dataset from directory at runtime
face_database_dict = load_face_templates("database")

Indexing dataset folder: database
 -> Indexed: Animesh (4 photos)
 -> Indexed: Sunny Singh (5 photos)
 -> Indexed: Tejrit Tyagi (5 photos)
 -> Indexed: Nivan Tyagi (5 photos)
 -> Indexed: Vimala (4 photos)
 -> Indexed: Shrinav Tyagi (5 photos)
 -> Indexed: Rakesh Ranjan (6 photos)
--- Indexing Complete ---



In [22]:
# 2. Test identification
identify_face_from_memory("Datasets/vimala.jpg", face_database_dict)

Result: Match found -> Vimala (Score: 0.9286)


'Vimala'

In [25]:
test_images = ['Tejrit.jpg', 'Shrinav.jpg', 'vimala.jpg', 'obama.jpg', 'group_photo.jpg']

for idx in test_images:
    identify_face_from_memory("Datasets/"+idx, face_database_dict)

Result: Match found -> Tejrit Tyagi (Score: 0.8793)
Result: Match found -> Shrinav Tyagi (Score: 0.8420)
Result: Match found -> Vimala (Score: 0.9286)
Result: Match found -> Unknown (Score: 0.1723)
Result: Match found -> Nivan Tyagi (Score: 0.8030)


In [ ]:
import os
import glob
import cv2
import numpy as np

# ==========================================
# 1. CONFIGURATION & MODEL INITIALIZATION
# ==========================================
DATASET_DIR = "database"
DETECTOR_MODEL = "Datasets/face_detection_yunet_2023mar.onnx"
RECOGNIZER_MODEL = "Datasets/face_recognition_sface_2021dec.onnx"

# Video matching threshold (slightly relaxed from 0.363 for real-world video noise)
VIDEO_SFACE_THRESHOLD = 0.33

# Working resolution for smooth edge inference
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# Initialize YuNet Face Detector with relaxed threshold for noisy/motion frames
detector = cv2.FaceDetectorYN.create(
    model=DETECTOR_MODEL,
    config="",
    input_size=(FRAME_WIDTH, FRAME_HEIGHT),
    score_threshold=0.5,   # Balanced between catching faces and avoiding false positives
    nms_threshold=0.3
)

# Initialize SFace Face Recognizer
recognizer = cv2.FaceRecognizerSF.create(
    model=RECOGNIZER_MODEL,
    config=""
)

# ==========================================
# 2. DATASET PARSING & TEMPLATE GENERATION
# ==========================================
def extract_embedding_from_file(img_path):
    """Loads an image, adjusts for scale, and extracts the 128D embedding."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    h_orig, w_orig, _ = img.shape
    target_width = 1000
    
    # Scale down oversized files (e.g., 4K phone captures) for detection reliability
    if w_orig > target_width:
        scale_factor = target_width / float(w_orig)
        img_small = cv2.resize(img, (target_width, int(h_orig * scale_factor)))
        detector.setInputSize((img_small.shape[1], img_small.shape[0]))
        _, faces = detector.detect(img_small)
        if faces is not None:
            faces[:, :14] = faces[:, :14] / scale_factor
    else:
        detector.setInputSize((w_orig, h_orig))
        _, faces = detector.detect(img)

    if faces is None:
        return None

    # Align and extract feature vector
    face_aligned = recognizer.alignCrop(img, faces)
    return recognizer.feature(face_aligned)

def load_dataset_templates(dataset_path):
    """Parses subfolders and computes normalized master embeddings."""
    templates = {}
    valid_exts = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.JPG', '*.JPEG', '*.PNG')
    
    if not os.path.exists(dataset_path):
        print(f"[-] Error: Directory '{dataset_path}' not found.")
        return templates

    print(f"[*] Indexing identities from '{dataset_path}'...")
    
    for person_name in sorted(os.listdir(dataset_path)):
        person_dir = os.path.join(dataset_path, person_name)
        if not os.path.isdir(person_dir):
            continue
            
        image_files = []
        for ext in valid_exts:
            image_files.extend(glob.glob(os.path.join(person_dir, ext)))
            
        if not image_files:
            print(f"    ! No valid photos for {person_name}. Skipping.")
            continue
            
        embeddings = []
        for img_file in image_files:
            emb = extract_embedding_from_file(img_file)
            if emb is not None:
                embeddings.append(emb)
                
        if embeddings:
            # Average multiple vectors to build a resilient representation
            master_vector = np.mean(embeddings, axis=0, dtype=np.float32)
            cv2.normalize(master_vector, master_vector)
            templates[person_name] = master_vector
            print(f"    -> Enrolled '{person_name}' ({len(embeddings)} image(s) processed)")
        else:
            print(f"    ! Warning: Failed to detect faces in all photos for '{person_name}'")
            
    print(f"[+] Indexing complete. {len(templates)} identities active in memory.\n")
    return templates

# ==========================================
# 3. LIVE STREAM PROCESSING LOOP
# ==========================================
def run_live_recognition():
    # 1. Load identities from folders
    templates = load_dataset_templates(DATASET_DIR)
    if not templates:
        print("[-] No templates loaded. Please ensure dataset/ has images.")
        return

    # 2. Reset detector input size to the video dimensions
    detector.setInputSize((FRAME_WIDTH, FRAME_HEIGHT))

    # 3. Initialize Camera Stream (0 for default webcam, or replace with video filename/RTSP link)
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

    if not cap.isOpened():
        print("[-] Error: Camera stream unavailable.")
        return

    print("[*] Starting live recognition. Press 'q' to exit.")
    
    frame_count = 0
    cached_detections = []  # Holds detection metadata to smooth out frame skipping

    while True:
        ret, frame = cap.read()
        if not ret:
            print("[-] Stream ended or frame read error.")
            break

        # Ensure frame matches inference shape
        frame_resized = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
        
        # Optimize system load: process detection and SFace every 2nd frame
        if frame_count % 2 == 0:
            cached_detections.clear()
            _, faces = detector.detect(frame_resized)

            if faces is not None:
                for face in faces:
                    # YuNet output layout: [0:4] bounding box, [4:14] landmarks, [14] score
                    single_face_input = np.array([face])
                    
                    try:
                        # Extract and match
                        aligned_crop = recognizer.alignCrop(frame_resized, single_face_input)
                        live_embedding = recognizer.feature(aligned_crop)

                        best_name = "Unknown"
                        max_score = -1.0

                        for name, master_emb in templates.items():
                            similarity = recognizer.match(live_embedding, master_emb, cv2.FaceRecognizerSF_FR_COSINE)
                            if similarity > max_score:
                                max_score = similarity
                                if similarity >= VIDEO_SFACE_THRESHOLD:
                                    best_name = name

                        box = face[0:4].astype(int)
                        cached_detections.append((box, best_name, max_score))
                    except Exception:
                        continue

        frame_count += 1

        # Render bounding boxes and identity labels
        for box, name, score in cached_detections:
            x, y, w, h = box
            is_recognized = (name != "Unknown")
            
            # Green for verified identity, red/amber for unknown
            color = (0, 255, 0) if is_recognized else (0, 0, 255)
            
            cv2.rectangle(frame_resized, (x, y), (x + w, y + h), color, 2)
            label = f"{name} ({score:.2f})" if score > 0 else name
            cv2.putText(
                frame_resized, 
                label, 
                (x, max(20, y - 8)), 
                cv2.FONT_HERSHEY_SIMPLEX, 
                0.55, 
                color, 
                2
            )

        cv2.imshow("SFace Live Recognition", frame_resized)

        # Press 'q' to gracefully shutdown
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("[*] Pipeline shut down cleanly.")

if __name__ == "__main__":
    run_live_recognition()
